# Pharmarize.ai - IndoBERT Fine-Tuning for Q&A

This notebook walks you through fine-tuning IndoBERT for Question Answering task.

**What is Q&A?** Given a question and a document/context, extract the answer directly from the context.

**Example:**
```
Question: "Apa kandungan pasak bumi?"
Context: "Tumbuhan pasak bumi mengandung eurycomanone yang bermanfaat untuk stamina."
Answer: "eurycomanone"
```

## Part 1: Setup & Data Preparation

In [ ]:
# Install required libraries (if not already installed)
import subprocess
import sys

# You should have these already from requirements.txt
# But just in case:
required_packages = [
    'transformers',
    'datasets',
    'torch',
    'pandas'
]

for package in required_packages:
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

: 

In [ ]:
# Import libraries
import json
import numpy as np
import pandas as pd
from pathlib import Path
import os
import logging
from datetime import datetime

# ML libraries
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset, DatasetDict

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {__import__('transformers').__version__}")

In [ ]:
# Configuration
class Config:
    # Model
    MODEL_NAME = "indobenchmark/indobert-base-p1"
    
    # Paths
    DATA_DIR = Path("../data")
    OUTPUT_DIR = Path("../models/checkpoints")
    FINAL_MODEL_DIR = Path("../models/pharmarize_qa_model")
    RESULTS_DIR = Path("../results")
    
    # Training
    BATCH_SIZE = 8
    LEARNING_RATE = 2e-5
    EPOCHS = 3
    WARMUP_STEPS = 100
    WEIGHT_DECAY = 0.01
    MAX_SEQ_LENGTH = 384
    
    # Device
    DEVICE = device
    
    # Seed for reproducibility
    SEED = 42

# Create directories
for dir_path in [Config.OUTPUT_DIR, Config.FINAL_MODEL_DIR, Config.RESULTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("Configuration loaded:")
print(f"  Model: {Config.MODEL_NAME}")
print(f"  Batch size: {Config.BATCH_SIZE}")
print(f"  Learning rate: {Config.LEARNING_RATE}")
print(f"  Epochs: {Config.EPOCHS}")
print(f"  Max sequence length: {Config.MAX_SEQ_LENGTH}")

## Part 2: Understanding SQuAD Format

The SQuAD (Stanford Question Answering Dataset) format is the standard for Q&A tasks.

**Structure:**
```json
{
  "data": [
    {
      "title": "Article Title",
      "paragraphs": [
        {
          "context": "Long text containing the answer...",
          "qas": [
            {
              "question": "What is..?",
              "id": "q1",
              "answers": [
                {
                  "text": "answer text",
                  "answer_start": 150  # Character position where answer starts
                }
              ],
              "is_impossible": false
            }
          ]
        }
      ]
    }
  ]
}
```

In [ ]:
# Let's create a SAMPLE dataset for demonstration
# In real use, you'll load from data/qa_dataset.json

sample_dataset = {
    "data": [
        {
            "title": "Tanaman Obat Indonesia",
            "paragraphs": [
                {
                    "context": "Tumbuhan pasak bumi mengandung eurycomanone yang bermanfaat untuk meningkatkan stamina dan vitalitas. Pasak bumi tumbuh di wilayah Indonesia bagian timur, khususnya di hutan-hutan tropis. Raja Limen adalah nama lain dari pasak bumi dalam bahasa lokal.",
                    "qas": [
                        {
                            "question": "Apa kandungan utama pasak bumi?",
                            "id": "q001",
                            "answers": [
                                {
                                    "text": "eurycomanone",
                                    "answer_start": 36
                                }
                            ],
                            "is_impossible": False
                        },
                        {
                            "question": "Di mana pasak bumi tumbuh?",
                            "id": "q002",
                            "answers": [
                                {
                                    "text": "Indonesia bagian timur",
                                    "answer_start": 130
                                }
                            ],
                            "is_impossible": False
                        }
                    ]
                }
            ]
        },
        {
            "title": "Kunyit",
            "paragraphs": [
                {
                    "context": "Kunyit (Curcuma longa) adalah rempah-rempah yang banyak digunakan dalam pengobatan tradisional Indonesia. Kandungan utama kunyit adalah kurkumin yang memiliki sifat anti-inflamasi. Kunyit biasa dikonsumsi dengan air hangat untuk menjaga kesehatan pencernaan.",
                    "qas": [
                        {
                            "question": "Apa nama ilmiah kunyit?",
                            "id": "q003",
                            "answers": [
                                {
                                    "text": "Curcuma longa",
                                    "answer_start": 7
                                }
                            ],
                            "is_impossible": False
                        },
                        {
                            "question": "Apa kandungan utama kunyit?",
                            "id": "q004",
                            "answers": [
                                {
                                    "text": "kurkumin",
                                    "answer_start": 152
                                }
                            ],
                            "is_impossible": False
                        }
                    ]
                }
            ]
        }
    ]
}

# Save sample dataset
sample_data_path = Config.DATA_DIR / "qa_dataset_sample.json"
with open(sample_data_path, 'w', encoding='utf-8') as f:
    json.dump(sample_dataset, f, indent=2, ensure_ascii=False)

print(f"✓ Sample dataset saved to {sample_data_path}")
print(f"\nDataset structure:")
print(f"  - 2 articles (Pasak Bumi, Kunyit)")
print(f"  - 4 Q&A pairs total")
print(f"\nFirst context preview:")
print(f"  {sample_dataset['data'][0]['paragraphs'][0]['context'][:100]}...")

## Part 3: Load & Preprocess Data

In [ ]:
def load_squad_dataset(json_path):
    """
    Load SQuAD format dataset and convert to HuggingFace Dataset format
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    
    # Extract Q&A pairs
    questions = []
    contexts = []
    answers = []
    answer_starts = []
    ids = []
    
    for article in raw_data['data']:
        for paragraph in article['paragraphs']:
            context = paragraph['context']
            
            for qa in paragraph['qas']:
                question = qa['question']
                qa_id = qa['id']
                
                # Get first answer (usually only one)
                if qa['answers']:
                    answer_text = qa['answers'][0]['text']
                    answer_start = qa['answers'][0]['answer_start']
                    
                    questions.append(question)
                    contexts.append(context)
                    answers.append(answer_text)
                    answer_starts.append(answer_start)
                    ids.append(qa_id)
    
    # Create dataset
    dataset = Dataset.from_dict({
        'id': ids,
        'question': questions,
        'context': contexts,
        'answers': [
            {'text': [ans], 'answer_start': [start]}
            for ans, start in zip(answers, answer_starts)
        ]
    })
    
    return dataset

# Load dataset
print("Loading dataset...")
dataset = load_squad_dataset(sample_data_path)
print(f"✓ Loaded {len(dataset)} Q&A pairs")
print(f"\nFirst example:")
print(f"  Q: {dataset[0]['question']}")
print(f"  A: {dataset[0]['answers']['text'][0]}")
print(f"  Context: {dataset[0]['context'][:80]}...")

In [ ]:
# Split into train/validation
# For real training: 70% train, 15% val, 15% test
# For small datasets like ours: use what we have

train_test_split = dataset.train_test_split(test_size=0.3, seed=Config.SEED)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {Config.MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
print(f"✓ Tokenizer loaded")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  Max model input: {tokenizer.model_max_length}")

In [ ]:
def preprocess_function(examples):
    """
    Tokenize and prepare data for Q&A model
    
    Key concept: The model needs to know WHERE in the tokenized sequence
    the answer starts and ends. We calculate this as token positions.
    """
    questions = [q.strip() for q in examples["question"]]
    contexts = [c.strip() for c in examples["context"]]
    answers = examples["answers"]
    
    # Tokenize with question + context together
    inputs = tokenizer(
        questions,
        contexts,
        max_length=Config.MAX_SEQ_LENGTH,
        truncation="only_second",  # Truncate context if too long, keep question
        return_offsets_mapping=True,  # Track character positions -> token positions
        padding="max_length"
    )
    
    offset_mapping = inputs.pop("offset_mapping")
    
    start_positions = []
    end_positions = []
    
    for i in range(len(questions)):
        # Find where answer starts/ends in character positions
        answer = answers[i]
        answer_start_char = answer["answer_start"][0]
        answer_text = answer["text"][0]
        answer_end_char = answer_start_char + len(answer_text)
        
        # Convert character positions to token positions
        sequence_ids = inputs.sequence_ids(i)
        context_start = sequence_ids.index(1)  # Where context starts (after [CLS] and question)
        context_end = len(sequence_ids) - 1
        while sequence_ids[context_end] != 1:
            context_end -= 1
        
        # If answer is fully in context
        if offset_mapping[i][context_start][0] <= answer_start_char and \
           offset_mapping[i][context_end][1] >= answer_end_char:
            
            # Find token indices of answer
            token_start_index = context_start
            while token_start_index <= context_end and \
                  offset_mapping[i][token_start_index][0] < answer_start_char:
                token_start_index += 1
            
            token_end_index = context_end
            while token_end_index >= context_start and \
                  offset_mapping[i][token_end_index][1] > answer_end_char:
                token_end_index -= 1
            
            start_positions.append(token_start_index)
            end_positions.append(token_end_index + 1)  # +1 because end is exclusive
        else:
            # Answer not found (shouldn't happen with good data)
            start_positions.append(0)
            end_positions.append(0)
    
    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    
    return inputs

# Apply preprocessing
print("Preprocessing data...")
train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)
eval_dataset = eval_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=eval_dataset.column_names
)

print(f"✓ Preprocessing complete")
print(f"\nTrain dataset shape:")
print(f"  {train_dataset}")
print(f"\nExample tokenized input:")
example = train_dataset[0]
print(f"  Input IDs length: {len(example['input_ids'])}")
print(f"  Start position: {example['start_positions']}")
print(f"  End position: {example['end_positions']}")

## Part 4: Load Model

In [ ]:
# Load pre-trained IndoBERT model for Question Answering
print(f"Loading model: {Config.MODEL_NAME}")
model = AutoModelForQuestionAnswering.from_pretrained(Config.MODEL_NAME)
print(f"✓ Model loaded")

# Show model size
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel statistics:")
print(f"  Total parameters: {total_params:,.0f}")
print(f"  Trainable parameters: {trainable_params:,.0f}")
print(f"  Model size: ~{total_params * 4 / 1e9:.1f} GB (in float32)")

## Part 5: Training Setup

### Key Training Concepts:

**Learning Rate**: How fast the model learns
- Too high: Model doesn't converge or overshoots
- Too low: Very slow training
- For fine-tuning: Usually 2e-5 (0.00002) is good

**Batch Size**: How many examples before updating weights
- Larger batch = more stable but slower
- Small batch = noisy but can train faster
- We use 8 to fit in 8-16GB RAM

**Epochs**: How many times to see the entire dataset
- 3-5 epochs typical for fine-tuning

**Warmup Steps**: Gradually increase learning rate at start
- Helps model stabilize
- ~10% of total steps

In [ ]:
# Setup training arguments
training_args = TrainingArguments(
    output_dir=str(Config.OUTPUT_DIR),
    evaluation_strategy="epoch",  # Evaluate after each epoch
    save_strategy="epoch",  # Save checkpoint after each epoch
    learning_rate=Config.LEARNING_RATE,
    per_device_train_batch_size=Config.BATCH_SIZE,
    per_device_eval_batch_size=Config.BATCH_SIZE,
    num_train_epochs=Config.EPOCHS,
    weight_decay=Config.WEIGHT_DECAY,
    warmup_steps=Config.WARMUP_STEPS,
    save_total_limit=2,  # Only keep best 2 checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    seed=Config.SEED
)

print("Training arguments configured:")
print(f"  Learning rate: {Config.LEARNING_RATE}")
print(f"  Batch size: {Config.BATCH_SIZE}")
print(f"  Epochs: {Config.EPOCHS}")
print(f"  Warmup steps: {Config.WARMUP_STEPS}")
print(f"  Total training steps: ~{len(train_dataset) // Config.BATCH_SIZE * Config.EPOCHS}")

In [ ]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    tokenizer=tokenizer
)

print("✓ Trainer created and ready for training")

## Part 6: Train the Model 🚀

In [ ]:
# Start training
print("Starting training...\n")
print(f"⏱️  This may take a few minutes on CPU")
print(f"Device: {Config.DEVICE}")

train_result = trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print(f"\nFinal training loss: {train_result.training_loss:.4f}")

## Part 7: Evaluate the Model

In [ ]:
# Evaluate on validation set
print("Evaluating on validation set...\n")
eval_result = trainer.evaluate()

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
for key, value in eval_result.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Save final model for inference
print(f"\nSaving final model to {Config.FINAL_MODEL_DIR}...")
model.save_pretrained(str(Config.FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(Config.FINAL_MODEL_DIR))

print("✓ Model saved successfully")
print(f"\nYou can now load it with:")
print(f"  from transformers import AutoModelForQuestionAnswering")
print(f"  model = AutoModelForQuestionAnswering.from_pretrained('{Config.FINAL_MODEL_DIR}')")

## Part 8: Test Inference

In [ ]:
# Load the fine-tuned model for inference
print("Loading fine-tuned model for inference...")
inference_model = AutoModelForQuestionAnswering.from_pretrained(str(Config.FINAL_MODEL_DIR))
inference_tokenizer = AutoTokenizer.from_pretrained(str(Config.FINAL_MODEL_DIR))
inference_model.to(Config.DEVICE)
inference_model.eval()

print("✓ Model loaded for inference")

In [ ]:
def predict_answer(question, context, model, tokenizer, device="cpu"):
    """
    Predict answer for a question given context
    """
    # Tokenize
    inputs = tokenizer.encode_plus(
        question,
        context,
        add_special_tokens=True,
        return_tensors="pt",
        max_length=384,
        truncation=True
    )
    
    # Move to device
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    
    # Get predictions
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
    
    # Extract answer span
    start_logits = outputs.start_logits
    end_logits = outputs.end_logits
    
    start_idx = torch.argmax(start_logits)
    end_idx = torch.argmax(end_logits)
    
    # Decode tokens to text
    tokens = inference_tokenizer.convert_ids_to_tokens(input_ids[0])
    answer_tokens = tokens[start_idx:end_idx+1]
    answer = inference_tokenizer.convert_tokens_to_string(answer_tokens)
    
    # Calculate confidence
    confidence = (float(start_logits[0, start_idx]) + float(end_logits[0, end_idx])) / 2
    
    return {
        "answer": answer.strip(),
        "confidence": confidence,
        "start_idx": int(start_idx),
        "end_idx": int(end_idx)
    }

# Test on training examples
print("Testing inference on training data:\n")
print("="*70)

for i in range(min(3, len(dataset))):
    example = dataset[i]
    question = example['question']
    context = example['context']
    true_answer = example['answers']['text'][0]
    
    # Get prediction
    result = predict_answer(question, context, inference_model, inference_tokenizer, Config.DEVICE)
    
    print(f"\nQuestion: {question}")
    print(f"Context: {context[:80]}...")
    print(f"\n✓ Predicted: {result['answer']}")
    print(f"• True answer: {true_answer}")
    print(f"• Confidence: {result['confidence']:.4f}")
    print(f"-" * 70)

## Part 9: Save Metrics & Training Summary

In [ ]:
# Save metrics to file
metrics = {
    "training_date": datetime.now().isoformat(),
    "model": Config.MODEL_NAME,
    "training_samples": len(train_dataset),
    "validation_samples": len(eval_dataset),
    "learning_rate": Config.LEARNING_RATE,
    "batch_size": Config.BATCH_SIZE,
    "epochs": Config.EPOCHS,
    "max_seq_length": Config.MAX_SEQ_LENGTH,
    "training_loss": float(train_result.training_loss),
    "eval_loss": float(eval_result.get('eval_loss', 0)),
}

metrics_path = Config.RESULTS_DIR / "training_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"✓ Metrics saved to {metrics_path}")

# Print summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
for key, value in metrics.items():
    if isinstance(value, (int, float)):
        if isinstance(value, float) and key.endswith('loss'):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")
    else:
        print(f"{key}: {value}")

print("\n" + "="*60)
print("Model saved to:")
print(f"  {Config.FINAL_MODEL_DIR}")
print("="*60)

## Summary & Next Steps

### What We Did:
1. ✅ Loaded & preprocessed SQuAD format Q&A data
2. ✅ Fine-tuned IndoBERT for extractive Question Answering
3. ✅ Evaluated the model on validation data
4. ✅ Tested inference on new examples
5. ✅ Saved the fine-tuned model

### Key Concepts Learned:
- **SQuAD format**: Question + Context + Answer with character positions
- **Tokenization**: Converting text to token IDs and tracking positions
- **Fine-tuning**: Training pre-trained model on task-specific data
- **Inference**: Using trained model to answer new questions

### Next Steps:
1. **Collect Real Data**: Download journals from Garba Rujukan Digital
2. **Create Q&A Dataset**: Convert journal text to SQuAD format
3. **Retrain**: Use this notebook with your real dataset
4. **Optimize**: Try different hyperparameters if needed
5. **Deploy**: Use the model in Phase 3 integration

### Tips for Better Results:
- **More data**: 100+ carefully created Q&A pairs
- **Balanced questions**: Mix different question types
- **Long contexts**: Test with realistic document lengths
- **Multiple epochs**: Sometimes helps with small datasets
- **Hyperparameter tuning**: Try learning_rate: 1e-5, 2e-5, 3e-5